In [1]:
samples = ["A1", "A2", "B2", "C2", "D1"]

In [4]:
import spatialdata as sd
import plotnine as p9
import scvi
import scanpy as sc

/Users/cydricgeyskens/miniconda3/envs/st-analysis/lib/python3.10/site-packages/scvi/__init__.py:31: DeprecationWarning: scvi is deprecated, please uninstall scvi via `pip uninstall scvi` and install the new scvi-tools package at github.com/YosefLab/scvi-tools


In [6]:
# loading zarr
sdata = sd.read_zarr("/Users/cydricgeyskens/Documents/code/phd/spatial-transcriptomics/scverse/hpc-scripts/intermediate_results/202601012.zarr")
sdata

PathNotFoundError: nothing found at path ''

In [4]:
# loading the model
scvi_model = scvi.model.SCVI.load("intermediate_results/scvi_model_20260112.pt")

INFO     File intermediate_results/scvi_model_20260112.pt/model.pt already downloaded                              


/data/leuven/357/vsc35768/miniconda3/envs/st-analysis/lib/python3.12/site-packages/scvi/model/base/_base_model.py:869: UserWarning: Save path contains no saved anndata and no adata was passed. Model will be loaded without anndata.


In [7]:
sdata.tables["D1_transcriptomics_filter_scvi_clusters"].obs["cell_type"].value_counts()

KeyError: 'cell_type'

In [7]:
adatas = []
for s in samples:
    a = sdata.tables[f"{s}_transcriptomics_filter_scvi_clusters"].copy()
    a.obs["sample"] = s
    adatas.append(a)

adata_scvi = ad.concat(adatas, join="outer")  

NameError: name 'ad' is not defined

## Differential Expression analysis

In [5]:
adata_scvi.obs["cell_type"].value_counts()

NameError: name 'adata_scvi' is not defined

In [ ]:
cell_type_1 = "Astrocytes Protoplasmic"
cell_idx1 = adata_scvi.obs["cell_type"] == cell_type_1
print(sum(cell_idx1), "cells of type", cell_type_1)

cell_type_2 = "DG Granule Cells"
cell_idx2 = adata_scvi.obs["cell_type"] == cell_type_2
print(sum(cell_idx2), "cells of type", cell_type_2)

In [ ]:
de_change = model_scvi.differential_expression(idx1=cell_idx1, idx2=cell_idx2, mode="change")
de_change.head(30)

In [ ]:
de_change_uniform = model_scvi.differential_expression(
    idx1=cell_idx1,  # we use the same cells as chosen before
    idx2=cell_idx2,
    weights="uniform",
    batch_correction=True,
    mode="change",
)
de_change_uniform.head(60)

In [ ]:
de_change_uniform["log10_pscore"] = np.log10(de_change_uniform["proba_not_de"])
de_change_uniform = de_change_uniform.join(adata.var, how="inner")
de_change_uniform.head(60)

In [ ]:
de_change_importance = model_scvi.differential_expression(
    idx1=cell_idx1,  # we use the same cells as chosen before
    idx2=cell_idx2,
    weights="importance",
    filter_outlier_cells=True,
    batch_correction=True,
    mode="change",
)

In [ ]:
de_change_importance["log10_pscore"] = np.log10(de_change_importance["proba_not_de"])
de_change_importance = de_change_importance.join(adata.var, how="inner")
de_change_importance.head(60)